# Tutorial 4 - Mouse stroke: a 2D and a 3D counterfactual prediction

Photothrombotic stroke in mouse cortex. Four sections were taken from each animal, and the
sections of one animal are registered into a shared frame, so a lesion learned on a stroke animal
can be **imposed on a sham animal** and the prediction compared against the real lesion in the
stroke animal.

### What is being predicted

The lesion has two compartments, and the model is given them as two **separate perturbation
labels**:

* **ICA** - the ischaemic core
* **PIA** - the peri-infarct area, the rim around it

The counterfactual is: *take a healthy (sham) brain, impose a lesion of the same shape, size and
per-section extent as the real one, and predict what each spot's expression becomes.* The
comparison is the measured stroke animal, which carries the real lesion in the same coordinate
frame - the same object measured twice.

### The two arms

| arm | footprint field | what it can express |
|---|---|---|
| `stroke_2d_cr` | `(x, y)`, sections superimposed | one spatial field shared by the four sections |
| `stroke_3d_cr` | `(x, y, ordinal section)` | a field that can differ between sections |

Both are trained with the sections tiled **by animal**, which is what puts the four sections of
one animal in a common frame. The 3D arm adds the section index as a third coordinate of the
footprint field.

> The `_cr` suffix matters. The arms **without** it (`stroke_2d`, `stroke_3d`) give the whole
> lesion a single perturbation label and therefore cannot express a core/rim difference at all -
> they are flat inside the lesion by construction. Only the `_cr` arms are used here.

### What the notebook does

1. resolve paths and open both checkpoints, and show what actually differs between them
2. explain the three decode passes and why assembling the two compartments from two passes is exact
3. optionally re-run those passes on a GPU; otherwise read the shipped decodes
4. assemble the predicted induction and put it beside the measured lesion, per section, for both arms
5. quantify: core against rim, and the rim-to-core depth profile
6. state what does **not** hold - three earlier readings of this experiment were withdrawn

## 0. Before you start

Sections 1-4 and 6 onward are **CPU-only**: they read the shipped decode outputs and draw. Only
section 5, which re-runs the decodes, needs a GPU (about 20 minutes per arm on one H100).

```bash
salloc -p gpu --gres=gpu:1 --mem=64G -c 8 -t 0-04:00
source /n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/miniconda3_new/etc/profile.d/conda.sh
conda activate /n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/concert_env
jupyter lab --no-browser --ip=0.0.0.0
```

## 1. Paths

The section coordinates are **already registered** before CONCERT sees them: `prep_stroke_h5ad.py`
reads a PASTE2 alignment from the registration delivery and writes it into the slide files. No
alignment happens inside the model. Overlap between paired sections varies from about 0.3 to 0.8,
which is worth remembering when reading a per-section panel.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

# The repository root: the directory holding models/ and benchmark_evaluation/. Found by
# walking up from the working directory, so the notebook runs from anywhere inside the repo;
# set CONCERT_CODEBASE to override. Every other root below reads an environment variable too,
# with the paths these results were produced on as the fallback -- see datasets/README.md.
def _repo_root():
    env = os.environ.get("CONCERT_CODEBASE")
    if env:
        return Path(env)
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "models" / "concert_benchmark").is_dir():
            return d
    raise RuntimeError("cannot locate the repository root; set CONCERT_CODEBASE")

def _root(var, default):
    return Path(os.environ.get(var, default))

CODEBASE  = _repo_root()
MODEL_SRC = CODEBASE / "models" / "concert_spac_niche"      # the copy that produced these results
EXP       = CODEBASE / "experiments" / "07_stroke_case_study"

NS      = _root("CONCERT_CASE_STUDY_ROOT", "/n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/CONCERT")
SLIDES  = NS / "stroke3d" / "slides_cr"                      # the compartment-labelled slides
OUTPUTS = _root("CONCERT_OUTPUT_ROOT", str(NS / "outputs"))
ARM_DIR = {"2d": OUTPUTS / "stroke_2d_cr", "3d": OUTPUTS / "stroke_3d_cr"}

# region labels of the measured stroke animal (ICA / PIA), from the registration delivery
DELIVERY = _root("CONCERT_STROKE_DELIVERY", "/n/holylfs06/LABS/mzitnik_lab/Lab/bof695/CONCERT/delivery/stroke-registration-benchmark")
REGION_H5 = DELIVERY / "benchmark_artifacts" / "data" / "Mouse_brain_stroke_all_data.h5"

WORKDIR = Path(os.environ.get(
    "CONCERT_NB_WORKDIR",
    "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/CONCERT/notebook_runs/04_stroke"))
WORKDIR.mkdir(parents=True, exist_ok=True)

STACKS = ["PT", "Sham"]          # PT = photothrombosis (real lesion), Sham = healthy control

checks = [("model source",        MODEL_SRC / "run_concert_spac.py"),
          ("figure code",         EXP / "figures" / "fig_cr_patch.py"),
          ("slides (sham 1)",     SLIDES / "stroke_paste2_Sham1.h5ad"),
          ("slides (stroke 1)",   SLIDES / "stroke_paste2_PT1.h5ad"),
          ("region labels",       REGION_H5)]
for arm, d in ARM_DIR.items():
    checks += [(f"{arm} checkpoint",   d / "model.pt"),
               (f"{arm} gene axis",    d / "hvg_gene_names.txt"),
               (f"{arm} lesion shape", d / "shape_sets" / "shape_core_local.txt"),
               (f"{arm} sham index",   d / "sham_cells.json"),
               (f"{arm} shipped stats", d / f"figure_stats_patch_{arm}_cr.json")]
for label, p in checks:
    print(f"{'OK ' if p.exists() else 'MISSING':>8}  {label:<20}  {p}")
print(f"\nnotebook output dir: {WORKDIR}")

## 2. What differs between the two checkpoints

Both arms are trained from the same script with a single flag changed:
`--fpert_n_z_levels 4` for the 3D arm against `1` for the 2D arm. Everything else - the slides,
the labels, the lengthscale band, the niche head - is identical.

The difference shows up in the weights as an extra input axis on the footprint field: the 3D
arm's inducing points and feature projection carry a third coordinate. The cell below reports the
tensors whose shapes differ, so the claim is checkable rather than asserted.

In [ ]:
import torch
import numpy as np

sd = {}
for arm, d in ARM_DIR.items():
    s = torch.load(str(d / "model.pt"), map_location="cpu", weights_only=False)
    sd[arm] = s.get("model_state_dict", s) if isinstance(s, dict) else s
    n = sum(v.numel() for v in sd[arm].values() if hasattr(v, "numel"))
    print(f"{arm}: {len(sd[arm])} tensors, {n:,} parameters")

print("\ntensors whose shape differs between the arms:")
for k in sorted(set(sd["2d"]) & set(sd["3d"])):
    a, b = sd["2d"][k], sd["3d"][k]
    if hasattr(a, "shape") and tuple(a.shape) != tuple(b.shape):
        print(f"  {k:<40} 2d {tuple(a.shape)}   3d {tuple(b.shape)}")
only2, only3 = set(sd["2d"]) - set(sd["3d"]), set(sd["3d"]) - set(sd["2d"])
print(f"\ntensors only in 2d: {sorted(only2) or 'none'}")
print(f"tensors only in 3d: {sorted(only3) or 'none'}")

print("\nperturbation vocabulary:", json.loads((ARM_DIR["2d"] / "pert_map.json").read_text())["pert_map"])

## 3. The counterfactual: three decode passes

The same cells are decoded three times, differing **only** in the perturbation attribute:

| pass | perturbation | applied to |
|---|---|---|
| `oncore` | `ICA` | the imposed **core** spots |
| `onrim` | `PIA` | the imposed **rim** spots |
| `off` | `__basal__` | every sham spot - the matched control |

The predicted induction is then assembled per compartment:

```
delta[core] = oncore[core] - off[core]
delta[rim]  = onrim[rim]  - off[rim]
```

**This assembly is exact, not an approximation.** The niche descriptor builds a spot's
microenvironment code from the expression latent and the coordinates only - it never reads its
neighbours' perturbation labels. So a core spot's decode is unaffected by whether the rim spots
were switched on in the same pass, and taking each compartment from the pass that switched it on
gives the same answer as a single pass with both on.

The imposed geometry is fixed by three index files under `shape_sets/`, transferred from the real
lesion so shape, size and per-section extent match:

* `shape_lesion_local.txt` - every spot inside the imposed lesion
* `shape_core_local.txt` - the core subset
* `shape_rim_local.txt` - the rim subset

(`_local` indexes the sham stack; the un-suffixed files index all eight slides and are what the
decode command is given.)

## 4. Running the decodes

`--stage eval` loads `<outdir>/model.pt`, so the output directory has to contain the checkpoint.
As in tutorial 3, the cell builds a **symlink shadow** of the arm directory under `WORKDIR` and
writes there, rather than writing to `/n/holylfs06`.

`RUN_DECODE` is off by default; the shipped decode outputs are read instead.

In [ ]:
RUN_DECODE = False        # True -> re-run the three passes for RUN_ARMS on a GPU
RUN_ARMS   = ["2d"]       # ["2d", "3d"] for both (about 20 min per arm)

SLIDE_FILES = ",".join(str(SLIDES / f"stroke_paste2_{s}{i}.h5ad")
                       for s in STACKS for i in (1, 2, 3, 4))
PASSES = [("oncore", "ICA",       "shape_core.txt"),
          ("onrim",  "PIA",       "shape_rim.txt"),
          ("off",    "__basal__", None)]              # None -> every sham spot

def decode_cmd(arm, shadow, pass_name, target, cells_file):
    nz = {"2d": "1", "3d": "4"}[arm]
    return [
        sys.executable, "-u", "run_concert_spac.py", "--stage", "eval",
        "--config", "config.yaml", "--sample", f"stroke_{arm}_cr_{pass_name}",
        "--multi_sample_files", SLIDE_FILES,
        "--outdir", str(shadow), "--model_file", "model.pt", "--project_index", "stroke",
        "--hvg_max_rank", "2000", "--seed", "42",
        "--unassigned_label", "unperturbed", "--control_label", "__NONE__",
        "--min_cells_per_pert", "25",
        "--use_action_correction", "False", "--use_pad", "False",
        "--inducing_point_steps", "6", "--grid_inducing_points", "True",
        "--loc_range", "20.0", "--kernel_scale", "10.0", "--multi_kernel_mode", "True",
        "--use_expr_encoder", "True", "--expr_hidden_dims", "512", "256",
        "--use_basal_encoder_only", "True",
        "--dynamicVAE", "False", "--free_bits", "0.5", "--gp_free_bits", "1.0",
        "--use_nngp", "False", "--use_paracrine", "False",
        "--use_niche_interaction", "False", "--use_niche_film", "True",
        "--niche_film_dim", "8", "--niche_film_k", "25",
        "--niche_film_gcap", "3.0", "--niche_film_l2", "1e-3",
        "--use_pert_gate", "True", "--gate_hidden", "64",
        "--use_f_pert", "True", "--f_pert_clamp", "5.0",
        "--spatial_basal_only", "True", "--lfc_ref_tissue", "CTX",
        "--basal_field", "rff", "--basal_bayes", "True",
        "--moran_lambda", "1.0", "--disentangle_radius", "4.0",
        "--fpert_field", "gp_pp", "--fpert_coord_scale", "20.0",
        "--fpert_rff_features", "128", "--fpert_rff_bandwidth", "0.15",
        "--fpert_ls_min", "0.05", "--fpert_ls_max", "0.60",
        "--tile_by_stack", "True", "--z_stack_obs", "mouse",
        "--z_index_obsm", "section_index",
        "--fpert_n_z_levels", nz, "--fpert_z_gain", "0.25",
        "--ls_prior_lambda", "1.0", "--ls_prior_min", "3", "--ls_prior_max", "18",
        "--fpert_svgp_inducing", "16", "--fpert_svgp_ls_init", "5.0",
        "--pert_cells", str(cells_file),
        "--target_cell_tissue", "CTX", "--target_cell_perturbation", target,
        "--verbosity", "1",
    ]

SHADOW = {}
for arm in ARM_DIR:
    sh = WORKDIR / f"{arm}_shadow"
    sh.mkdir(parents=True, exist_ok=True)
    for f in ARM_DIR[arm].iterdir():                 # checkpoint + geometry + sidecar files
        tgt = sh / f.name
        if not tgt.exists():
            tgt.symlink_to(f)
    SHADOW[arm] = sh

if RUN_DECODE:
    for arm in RUN_ARMS:
        for pass_name, target, shape_file in PASSES:
            cells_file = (SHADOW[arm] / "shape_sets" / shape_file) if shape_file \
                         else (SHADOW[arm] / "sham_cells.txt")
            print(f"\n=== {arm} / {pass_name}: {target} on {cells_file.name} ===")
            r = subprocess.run(decode_cmd(arm, SHADOW[arm], pass_name, target, cells_file),
                               cwd=str(MODEL_SRC))
            print("exit code", r.returncode)
else:
    print("RUN_DECODE is False -- the shipped decodes are read below.\n")
    for arm in ARM_DIR:
        got = sorted(p.name for p in ARM_DIR[arm].glob("*_perturbed_counts.h5ad"))
        print(f"{arm}: {len(got)} decode outputs on disk")
        for g in got[:4]:
            print("   ", g)

## 5. Assemble the predicted induction

Each decode writes one `*_perturbed_counts.h5ad` holding all eight slides in file order. Two
bookkeeping steps turn those into a per-compartment induction map:

1. **select the sham rows.** `sham_cells.json` records the per-file cell counts and which files
   belong to the sham animal, which gives the row block to keep.
2. **put the counts on a comparable scale.** Everything is compared as `log1p` of counts per
   10,000, on both the predicted and the measured side.

The measured side is the stroke animal's own expression, and its region labels (ICA / PIA) come
from the registration delivery rather than from the model.

In [ ]:
import anndata as ad, h5py, pandas as pd
from scipy.spatial import cKDTree

def log1p_cpm(X):
    s = X.sum(1, keepdims=True); s[s == 0] = 1.0
    return np.log1p(X / s * 1e4)

def load_stack(stack):
    ps = sorted(SLIDES.glob(f"stroke_paste2_{stack}*.h5ad"))
    A = ad.concat([ad.read_h5ad(p) for p in ps], label="sf", keys=[p.stem for p in ps],
                  index_unique=None)
    section = np.unique(np.asarray(A.obsm["section_index"]).reshape(-1), return_inverse=True)[1]
    return A, np.asarray(A.obsm["array_coords"], np.float32), section

def norm_xy(xy):
    lo, hi = xy.min(0), xy.max(0)
    return (xy - lo) / np.maximum(hi - lo, 1e-8)

def build_delta(arm, source_dir=None):
    """Predicted induction per spot on the sham animal, assembled per compartment."""
    d = Path(source_dir or ARM_DIR[arm])
    side = json.loads((d / "sham_cells.json").read_text())
    offsets = np.concatenate([[0], np.cumsum(side["counts"])])[:-1]
    rows = np.concatenate([np.arange(o, o + c)
                           for o, c, t in zip(offsets, side["counts"], side["is_target"]) if t])
    def read(pass_name):
        # The `_stroke_` segment is load-bearing: the arm directories also hold decodes from
        # the rim-thickness and observed-shape variants, named `onrim_obs_*`, `onrim_t1_*` and
        # so on. A looser glob picks one of those instead, in an order that is not defined.
        hits = sorted(d.glob(f"stroke_{arm}_cr_{pass_name}_stroke_*_perturbed_counts.h5ad"))
        if len(hits) != 1:
            raise FileNotFoundError(f"expected one {pass_name} decode in {d}, found {hits}")
        return log1p_cpm(np.asarray(ad.read_h5ad(hits[0]).X[rows], np.float32))
    L_core, L_rim, L_off = read("oncore"), read("onrim"), read("off")

    genes = np.loadtxt(d / "hvg_gene_names.txt", dtype=str)
    n_sham = L_off.shape[0]
    inside = lambda f: np.isin(np.arange(n_sham), np.loadtxt(d / "shape_sets" / f, dtype=np.int64))
    p_in   = inside("shape_lesion_local.txt")
    p_core = inside("shape_core_local.txt") & p_in
    p_rim  = inside("shape_rim_local.txt") & p_in

    delta = np.zeros_like(L_off)
    delta[p_core] = L_core[p_core] - L_off[p_core]
    delta[p_rim]  = L_rim[p_rim]  - L_off[p_rim]
    return dict(delta=delta, genes=genes, in_lesion=p_in, core=p_core, rim=p_rim)

PRED = {arm: build_delta(arm) for arm in ARM_DIR}
for arm, p in PRED.items():
    print(f"{arm}: {p['delta'].shape[0]} sham spots x {p['delta'].shape[1]} genes; "
          f"imposed lesion {int(p['in_lesion'].sum())} spots "
          f"({int(p['core'].sum())} core, {int(p['rim'].sum())} rim)")

In [ ]:
# the measured side: the stroke animal, and its region labels
A_pt,  xy_pt,  sec_pt  = load_stack("PT")
A_sh,  xy_sh,  sec_sh  = load_stack("Sham")
L_pt   = log1p_cpm(np.asarray(A_pt.X.todense(), np.float32))
L_sh   = log1p_cpm(np.asarray(A_sh.X.todense(), np.float32))   # healthy baseline, same places
genes_obs = np.asarray(A_pt.var_names)

with h5py.File(REGION_H5, "r") as f:
    bc  = np.array([b.decode() for b in f["Barcode"][:]])
    reg = np.array([b.decode() for b in f["Celltype_coarse"][:]])
region = pd.Series(reg, index=bc).loc[np.asarray(A_pt.obs_names)].to_numpy()
o_core, o_rim = region == "ICA", region == "PIA"
o_in = o_core | o_rim

print(f"stroke animal: {A_pt.n_obs} spots over {sec_pt.max() + 1} sections; "
      f"measured lesion {int(o_in.sum())} spots ({int(o_core.sum())} ICA, {int(o_rim.sum())} PIA)")
print(f"sham animal  : {A_sh.n_obs} spots over {sec_sh.max() + 1} sections")

## 6. Observed beside predicted

Nothing is shown as a prediction alone where a measurement exists. Each figure has four rows:

1. the measured expression in the stroke animal,
2. the predicted induction on the sham animal,
3. the measured lesion regions,
4. the imposed lesion regions.

The 2D arm sees each section independently, so it is drawn for **one** section: laying out four
would invite a reading across sections that the arm cannot support. The 3D arm gets all four.

In [ ]:
import matplotlib
sys.path.insert(0, str(CODEBASE / "experiments" / "03_dss_gut_case_study" / "figures"))
import paper_style as ps
ps.use()
# paper_style selects the Agg backend on import; switch back so the figures draw here.
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

CORE_C, RIM_C, OTHER_C = "#b2182b", "#ef8a62", "#dcdcdc"
npt, nsh = norm_xy(xy_pt), norm_xy(xy_sh)

def spatial_figure(arm, gene, sections=None):
    P = PRED[arm]
    if gene not in set(genes_obs.tolist()) or gene not in set(P["genes"].tolist()):
        raise KeyError(f"{gene} is not on both gene axes")
    ov = L_pt[:, int(np.where(genes_obs == gene)[0][0])]           # measured expression
    pv = P["delta"][:, int(np.where(P["genes"] == gene)[0][0])]    # predicted induction
    keep = sections if sections is not None else list(range(int(sec_pt.max()) + 1))

    reg_cmap = ListedColormap([OTHER_C, RIM_C, CORE_C])
    o_reg = np.where(o_core, 2, np.where(o_rim, 1, 0))
    p_reg = np.where(P["core"], 2, np.where(P["rim"], 1, 0))
    vo, vp = np.quantile(ov, [.03, .97]), np.quantile(pv, [.03, .97])

    fig, ax = plt.subplots(4, len(keep), figsize=(2.5 * len(keep) + 2.0, 10.0), squeeze=False)
    for ci, s in enumerate(keep):
        mp, ms = sec_pt == s, sec_sh == s
        h0 = ax[0, ci].scatter(npt[mp, 0], npt[mp, 1], c=ov[mp], s=2.6, lw=0,
                               cmap=ps.FIELD_CMAP, vmin=vo[0], vmax=vo[1], rasterized=True)
        h1 = ax[1, ci].scatter(nsh[ms, 0], nsh[ms, 1], c=pv[ms], s=2.6, lw=0,
                               cmap=ps.FIELD_CMAP, vmin=vp[0], vmax=vp[1], rasterized=True)
        ax[2, ci].scatter(npt[mp, 0], npt[mp, 1], c=o_reg[mp], s=2.6, lw=0,
                          cmap=reg_cmap, vmin=0, vmax=2, rasterized=True)
        ax[3, ci].scatter(nsh[ms, 0], nsh[ms, 1], c=p_reg[ms], s=2.6, lw=0,
                          cmap=reg_cmap, vmin=0, vmax=2, rasterized=True)
        for r in range(4):
            ax[r, ci].set_xticks([]); ax[r, ci].set_yticks([])
            ax[r, ci].set_aspect("equal"); ax[r, ci].grid(False)
            for sp in ax[r, ci].spines.values():
                sp.set_visible(False)
        ax[3, ci].set_xlabel(f"Section {s + 1}", fontsize=13)
    for r, lab in enumerate(["Measured\n(stroke animal)", "Predicted\n(sham animal)",
                             "Measured regions", "Imposed regions"]):
        ax[r, 0].set_ylabel(lab, fontsize=12, labelpad=6)
    fig.subplots_adjust(wspace=0.02, hspace=0.02, left=0.12, right=0.87, top=0.99, bottom=0.06)
    for h, arow, lb in ((h0, ax[0, -1], f"measured {gene}"), (h1, ax[1, -1], "predicted induction")):
        po = arow.get_position(); pad = 0.12 * po.height
        cax = fig.add_axes([0.885, po.y0 + pad, 0.016, po.height - 2 * pad])
        cb = fig.colorbar(h, cax=cax); cb.set_label(lb, fontsize=11, labelpad=2)
        cb.ax.tick_params(labelsize=9); cb.set_ticks([cb.vmin, cb.vmax])
    return fig

GENE = "Gm42418"
spatial_figure("2d", GENE, sections=[0]); plt.show()

In [ ]:
spatial_figure("3d", GENE); plt.show()

## 7. Core against rim, and the depth profile

The spatial panels show the lesion is reproduced in the right place. The quantitative question is
whether the **compartments** are told apart - whether the model puts a different response in the
core than in the rim.

Two readouts:

* **core and rim means.** For the prediction this is the mean induction in each compartment. For
  the measurement it is the stroke animal's expression in each region minus the sham animal's
  expression in the corresponding imposed region - an induction over healthy tissue, so the two
  sides are the same quantity.
* **the depth profile.** Spots inside the lesion are binned by distance from its boundary, from
  rim to core, and the induction is averaged per bin.

In [ ]:
NB = 6      # depth bins, rim -> core

def depth_of(xy, inside):
    d = np.zeros(len(xy))
    if inside.sum() and (~inside).sum():
        d[inside] = cKDTree(xy[~inside]).query(xy[inside])[0]
    return d

def profile(depth, v, inside, nb=NB):
    d, x = depth[inside], v[inside]
    q = np.quantile(d, np.linspace(0, 1, nb + 1)); q[-1] += 1e-6
    b = [x[(d >= q[i]) & (d < q[i + 1])] for i in range(nb)]
    return (np.array([bb.mean() for bb in b]),
            np.array([bb.std() / max(np.sqrt(len(bb)), 1) for bb in b]))

def compartment_stats(arm, gene):
    P = PRED[arm]
    gi_o = int(np.where(genes_obs == gene)[0][0])
    gi_p = int(np.where(P["genes"] == gene)[0][0])
    ov, pv = L_pt[:, gi_o], P["delta"][:, gi_p]
    base = L_sh[:, gi_o]                                   # healthy tissue, same places
    return {
        "gene": gene, "arm": arm,
        "observed_core":  float(ov[o_core].mean() - base[P["core"]].mean()),
        "observed_rim":   float(ov[o_rim].mean()  - base[P["rim"]].mean()),
        "predicted_core": float(pv[P["core"]].mean()),
        "predicted_rim":  float(pv[P["rim"]].mean()),
    }

dep_pred = depth_of(xy_sh, PRED["2d"]["in_lesion"])        # geometry is identical across arms
dep_obs  = depth_of(xy_pt, o_in)

GENES = ["Gm42418", "Scgb1a1", "Spp1", "Vim", "Lgals1", "Timp1"]
GENES = [g for g in GENES if g in set(genes_obs.tolist()) and g in set(PRED["2d"]["genes"].tolist())]

stats = pd.DataFrame([compartment_stats(a, g) for g in GENES for a in ("2d", "3d")])
stats["observed_core_minus_rim"]  = stats.observed_core - stats.observed_rim
stats["predicted_core_minus_rim"] = stats.predicted_core - stats.predicted_rim
stats["sign_matches"] = np.sign(stats.observed_core_minus_rim) == np.sign(stats.predicted_core_minus_rim)
display(stats.round(3))

for arm in ("2d", "3d"):
    s = stats[stats.arm == arm]
    print(f"{arm}: core-vs-rim direction matched on {int(s.sign_matches.sum())}/{len(s)} genes")

In [ ]:
CORE_GENES = [g for g in ["Gm42418", "Scgb1a1"] if g in GENES]
fig, axes = plt.subplots(2, len(CORE_GENES) * 2, figsize=(3.2 * len(CORE_GENES) * 2, 6.4),
                         squeeze=False)
xs = np.arange(NB)
col = 0
for arm in ("2d", "3d"):
    P = PRED[arm]
    for g in CORE_GENES:
        gi_o = int(np.where(genes_obs == g)[0][0])
        gi_p = int(np.where(P["genes"] == g)[0][0])
        od, oe = profile(dep_obs, L_pt[:, gi_o], o_in)
        od = od - float(L_sh[:, gi_o][P["in_lesion"]].mean())        # induction over healthy
        pdp, pe = profile(dep_pred, P["delta"][:, gi_p], P["in_lesion"])

        a0 = axes[0, col]
        a0.errorbar(xs, od, yerr=oe, color=ps.ACCENT, lw=2.0, marker="o", ms=5, label="measured")
        a0.errorbar(xs, pdp, yerr=pe, color="#2166ac", lw=2.0, marker="s", ms=5, ls="--",
                    label="CONCERT")
        a0.set_xticks([0, NB - 1]); a0.set_xticklabels(["Rim", "Core"], fontsize=12)
        a0.set_xlabel(f"{g}  ({arm})", fontsize=12)
        if col == 0:
            a0.set_ylabel("induction over healthy", fontsize=11); a0.legend(fontsize=9)

        st = compartment_stats(arm, g)
        a1 = axes[1, col]
        w, x2 = 0.36, np.arange(2)
        a1.bar(x2 - w / 2, [st["observed_core"], st["observed_rim"]], w, color=ps.ACCENT,
               label="measured")
        a1.bar(x2 + w / 2, [st["predicted_core"], st["predicted_rim"]], w, color="#2166ac",
               label="CONCERT")
        a1.set_xticks(x2); a1.set_xticklabels(["Core", "Rim"], fontsize=12)
        a1.set_xlabel(f"{g}  ({arm})", fontsize=12)
        if col == 0:
            a1.set_ylabel("mean induction", fontsize=11); a1.legend(fontsize=9)
        col += 1
fig.subplots_adjust(wspace=0.45, hspace=0.32)
plt.show()

## 8. Against the shipped figure statistics

`figure_stats_patch_<arm>_cr.json` was written by `figures/fig_cr_patch.py` on the same decodes.
Reproducing it is the check that the assembly above is the same computation.

In [ ]:
rows = []
for arm in ("2d", "3d"):
    shipped = json.loads((ARM_DIR[arm] / f"figure_stats_patch_{arm}_cr.json").read_text())["genes"]
    for g in CORE_GENES:
        if g not in shipped:
            continue
        here = compartment_stats(arm, g)
        for k in ("observed_core", "observed_rim", "predicted_core", "predicted_rim"):
            rows.append({"arm": arm, "gene": g, "quantity": k,
                         "shipped": shipped[g][k], "recomputed": here[k],
                         "abs_diff": abs(shipped[g][k] - here[k])})
chk = pd.DataFrame(rows)
print(f"largest absolute difference: {chk.abs_diff.max():.2e}")
chk.round(4)

## What holds, and what does not

Three earlier readings of this experiment were withdrawn after being measured properly. They are
listed here because each is an easy conclusion to reach again.

**What holds**

* The imposed lesion reproduces the measured one in **location and extent**, in both arms.
* The `_cr` arms get the **core-versus-rim direction** right on most of the genes tested; the 2D
  arm reaches 6 of 7 in the shipped run, including a rim-high gene.

**What does not hold**

* **The core/rim contrast comes from the perturbation label, not from the spatial field.** The
  single-label arms (`stroke_2d`, `stroke_3d`) are flat inside the lesion - the shipped numbers
  are core 0.801 against rim 0.789 for one marker. Three explanations for that flatness were
  proposed and all three were refuted by measurement: niche-descriptor dilution (the descriptor
  reads neighbours' latent states, never their perturbation labels), an over-long lengthscale
  (PIA's is 0.0878, about a quarter of the lesion), and vanishing features (the maximum feature
  runs 0.60-0.98, never near zero). A finer inducing grid raises within-lesion field variation
  nineteen-fold and still creates no core/rim contrast.
* **A lesion cannot propagate into unperturbed tissue in this model.** `f_pert` is gated by
  `is_real_KO`, so an unperturbed spot receives exactly zero. Any test of propagation to
  unperturbed tissue or to an unperturbed section returns 0.0000 in both arms and always will.
  The 3D kernel is the section index used as a third coordinate - it is not a propagation model.
* **Rim thickness is not a lever.** The imposed rim already matches the measured one (6.87
  against 6.63 spot-rings). Dilating it changes nothing per spot; the apparent decay of an
  observed-shaped rim is a coverage artefact, not a weaker response.
* **In the condition panels elsewhere in this case study, paracrine spillover was the entire
  injury signal.** With it off the response went to zero (one marker moved from +2.91 to -0.06).
  Bounding and normalising it did not help; only removing it did. The lesion-size dose-response
  disappeared with it, and that is correct - per-spot induction is close to size-invariant, and
  only the integrated response scales with lesion size.

**Still open**

The 3D core/rim result is unresolved. The 2D `_cr` arm reaches 6 of 7 signs; the best 3D arm
reached 5 of 6 in a single run, and across nine arms the score only moves between 3 of 6 and 5 of
6 with which genes pass. **Replicate across seeds before making any 3D core/rim claim.**

## Where things are

| | |
|---|---|
| model source | `codebase/models/concert_spac_niche/` |
| prep, figures, diagnostics | `codebase/experiments/07_stroke_case_study/` |
| the figure this notebook follows | `experiments/07_stroke_case_study/figures/fig_cr_patch.py` |
| checkpoints and decodes | `outputs/stroke_2d_cr/`, `outputs/stroke_3d_cr/` |